###### Imports and Settings

In [126]:
import pandas as pd
import geopandas as gpd
import numpy as np
import requests
import io
import pickle
import matplotlib.pyplot as plt
from collections import deque
from functools import reduce
%matplotlib inline
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)
pd.set_option('display.width', 150)

In [127]:
data = pd.read_csv('../data/ACS20205YR.csv')

# Page by Page of CC Databook to start:

## Population + Projections Summary:

In [128]:
data['Population'] = data['agebysex_total_series']

In [129]:
data = data.drop(columns = 'agebysex_total_series')

### Population Density  
Bring in geofiles

## Health Insurance Coverage  
+ Total Series  
+ Total No Health Insurance Coverage  
+ % Total with No Health Insurance Coverage  
+ Total with Health Insurance Coverage 
+ % Total with Health Insurance Coverage
+ Total with Health Insurance Coverage: Public Health Coverage  
+ % Covered Population: Public Health Coverage  
+ Total with Health Insurance Coverage: Private Health Coverage  
+ % Covered Population: Private Health Coverage  

In [130]:
#total series
data['Total Health Coverage Series'] = data['healthcoverage_total_healthcare_series']
data = data.drop(columns = 'healthcoverage_total_healthcare_series')

In [131]:
#total no health insurance
colsnohealthcare = ['healthcoverage_mu6_wo','healthcoverage_m6to18_wo','healthcoverage_m19to25_wo','healthcoverage_m26to34_wo','healthcoverage_m35to44_wo',
        'healthcoverage_m45to54_wo','healthcoverage_m55to64_wo','healthcoverage_m65to74_wo','healthcoverage_m75+_wo','healthcoverage_fu6_wo',
        'healthcoverage_f6to18_wo','healthcoverage_f19to25_wo','healthcoverage_f26to34_wo','healthcoverage_f35to44_wo','healthcoverage_f45to54_wo',
        'healthcoverage_f55to64_wo','healthcoverage_f65to74_wo','healthcoverage_f75+_wo']
nohealthcare = [data['healthcoverage_mu6_wo'],data['healthcoverage_m6to18_wo'],data['healthcoverage_m19to25_wo'],data['healthcoverage_m26to34_wo'],
                data['healthcoverage_m35to44_wo'],data['healthcoverage_m45to54_wo'],data['healthcoverage_m55to64_wo'],data['healthcoverage_m65to74_wo'],
                data['healthcoverage_m75+_wo'],data['healthcoverage_fu6_wo'],data['healthcoverage_f6to18_wo'],data['healthcoverage_f19to25_wo'],
                data['healthcoverage_f26to34_wo'],data['healthcoverage_f35to44_wo'],data['healthcoverage_f45to54_wo'],data['healthcoverage_f55to64_wo'],
                data['healthcoverage_f65to74_wo'],data['healthcoverage_f75+_wo']]
data['Total No Health Insurance Coverage'] = sum(nohealthcare)
data = data.drop(columns = colsnohealthcare)

In [132]:
#% of total with No Health Insurance Coverage
data['Population without Health Insurance %'] = data['Total No Health Insurance Coverage']/data['Total Health Coverage Series']

In [133]:
#total with health insurance
colshealthcare = ['healthcoverage_mu6_w','healthcoverage_m6to18_w','healthcoverage_m19to25_w','healthcoverage_m26to34_w','healthcoverage_m35to44_w',
        'healthcoverage_m45to54_w','healthcoverage_m55to64_w','healthcoverage_m65to74_w','healthcoverage_m75+_w','healthcoverage_fu6_w',
        'healthcoverage_f6to18_w','healthcoverage_f19to25_w','healthcoverage_f26to34_w','healthcoverage_f35to44_w','healthcoverage_f45to54_w',
        'healthcoverage_f55to64_w','healthcoverage_f65to74_w','healthcoverage_f75+_w']
healthcare = [data['healthcoverage_mu6_w'],data['healthcoverage_m6to18_w'],data['healthcoverage_m19to25_w'],data['healthcoverage_m26to34_w'],
                data['healthcoverage_m35to44_w'],data['healthcoverage_m45to54_w'],data['healthcoverage_m55to64_w'],data['healthcoverage_m65to74_w'],
                data['healthcoverage_m75+_w'],data['healthcoverage_fu6_w'],data['healthcoverage_f6to18_w'],data['healthcoverage_f19to25_w'],
                data['healthcoverage_f26to34_w'],data['healthcoverage_f35to44_w'],data['healthcoverage_f45to54_w'],data['healthcoverage_f55to64_w'],
                data['healthcoverage_f65to74_w'],data['healthcoverage_f75+_w']]
data['Total With Health Insurance Coverage'] = sum(healthcare)
data = data.drop(columns = colshealthcare)

In [134]:
#% of total with Health Insurance Coverage
data['Population with Health Insurance %'] = data['Total With Health Insurance Coverage']/data['Total Health Coverage Series']

In [135]:
# Total with Health Insurance: Public Coverage
colspublic = ['healthcoverage_mu6_wpublic','healthcoverage_m6to18_wpublic','healthcoverage_m19to25_wpublic','healthcoverage_m26to34_wpublic',
              'healthcoverage_m35to44_wpublic','healthcoverage_m45to54_wpublic','healthcoverage_m55to64_wpublic','healthcoverage_m65to74_wpublic',
              'healthcoverage_m75+_wpublic','healthcoverage_fu6_wpublic','healthcoverage_f6to18_wpublic','healthcoverage_f19to25_wpublic',
              'healthcoverage_f26to34_wpublic','healthcoverage_f35to44_wpublic','healthcoverage_f45to54_wpublic','healthcoverage_f55to64_wpublic',
              'healthcoverage_f65to74_wpublic','healthcoverage_f75+_wpublic']
public = [data['healthcoverage_mu6_wpublic'],data['healthcoverage_m6to18_wpublic'],data['healthcoverage_m19to25_wpublic'],data['healthcoverage_m26to34_wpublic'],
          data['healthcoverage_m35to44_wpublic'],data['healthcoverage_m45to54_wpublic'],data['healthcoverage_m55to64_wpublic'],
          data['healthcoverage_m65to74_wpublic'],data['healthcoverage_m75+_wpublic'],data['healthcoverage_fu6_wpublic'],
          data['healthcoverage_f6to18_wpublic'],data['healthcoverage_f19to25_wpublic'],data['healthcoverage_f26to34_wpublic'],
          data['healthcoverage_f35to44_wpublic'],data['healthcoverage_f45to54_wpublic'],data['healthcoverage_f55to64_wpublic'],
          data['healthcoverage_f65to74_wpublic'],data['healthcoverage_f75+_wpublic']]
data['Total With Health Insurance, Public'] = sum(public)
data = data.drop(columns = colspublic)

In [136]:
# % of covered with public
data['Covered, % with Public Health Insurance'] = data['Total With Health Insurance, Public']/data['Total With Health Insurance Coverage']

In [137]:
# Total with Health Insurance: Private Coverage
colsprivate = ['healthcoverage_mu6_wprivate','healthcoverage_m6to18_wprivate','healthcoverage_m19to25_wprivate','healthcoverage_m26to34_wprivate',
              'healthcoverage_m35to44_wprivate','healthcoverage_m45to54_wprivate','healthcoverage_m55to64_wprivate','healthcoverage_m65to74_wprivate',
              'healthcoverage_m75+_wprivate','healthcoverage_fu6_wprivate','healthcoverage_f6to18_wprivate','healthcoverage_f19to25_wprivate',
              'healthcoverage_f26to34_wprivate','healthcoverage_f35to44_wprivate','healthcoverage_f45to54_wprivate','healthcoverage_f55to64_wprivate',
              'healthcoverage_f65to74_wprivate','healthcoverage_f75+_wprivate']
private = [data['healthcoverage_mu6_wprivate'],data['healthcoverage_m6to18_wprivate'],data['healthcoverage_m19to25_wprivate'],data['healthcoverage_m26to34_wprivate'],
          data['healthcoverage_m35to44_wprivate'],data['healthcoverage_m45to54_wprivate'],data['healthcoverage_m55to64_wprivate'],
          data['healthcoverage_m65to74_wprivate'],data['healthcoverage_m75+_wprivate'],data['healthcoverage_fu6_wprivate'],
          data['healthcoverage_f6to18_wprivate'],data['healthcoverage_f19to25_wprivate'],data['healthcoverage_f26to34_wprivate'],
          data['healthcoverage_f35to44_wprivate'],data['healthcoverage_f45to54_wprivate'],data['healthcoverage_f55to64_wprivate'],
          data['healthcoverage_f65to74_wprivate'],data['healthcoverage_f75+_wprivate']]
data['Total With Health Insurance, Private'] = sum(private)
data = data.drop(columns = colsprivate)

In [138]:
# % of covered with private
data['Covered, % with Private Health Insurance'] = data['Total With Health Insurance, Private']/data['Total With Health Insurance Coverage']

## Age Sex Race Summary

### Race/Ethnicity #s   
+ White Alone  
+ Black/African American Alone  
+ American Indian Alaska Native Alone  
+ Asian Alone  
+ Native Hawaiian/Other Pacific Islander Alone  
+ Some Other Race Alone  
+ Two or More Races  
+ Hispanic/Latino  
+ White, Not Hispanic/Latino  
+ Total Minority (non-white, non Hispanic/Latino)

In [139]:
#White Alone
data['White Alone'] = data['raceeth_white_alone']
data['White Alone %'] = data['White Alone']/data['Population']
#Black or African American Alone 
data['Black or African American Alone'] = data['raceeth_blackafricanamerican_alone']
data['Black or African American Alone %'] = data['Black or African American Alone']/data['Population']
#American Indian and Alaska Native Alone
data['American Indian Alaska Native Alone'] = data['raceeth_americanindianalaskanative_alone']
data['American Indian Alaska Native Alone %'] = data['American Indian Alaska Native Alone']/data['Population']
#Asian Alone
data['Asian Alone'] = data['raceeth_asian_alone']
data['Asian Alone %'] = data['Asian Alone']/data['Population']
#Native Hawaiian Other Pacific Islander Alone
data['Native Hawaiian Other Pacific Islander Alone'] = data['raceeth_nativehawaiianotherpacificislander_alone']
data['Native Hawaiian Other Pacific Islander Alone %'] = data['Native Hawaiian Other Pacific Islander Alone']/data['Population']
#Some Other Race Alone
data['Some Other Race Alone'] = data['raceeth_someotherrace_alone']
data['Some Other Race Alone %'] = data['Some Other Race Alone']/data['Population']
#Two or More Races
data['Two or More Races'] = data['raceeth_twoormoreraces_alone']
data['Two or More Races %'] = data['Two or More Races']/data['Population']
#Hispanic or Latino
data['Hispanic or Latino'] = data['raceeth_hispanicorlatino']
data['Hispanic or Latino %'] = data['Hispanic or Latino']/data['Population']
#Data Minority
data['Minority'] = data['Population'] - data['raceeth_whitealone_nothispanicorlatino']
data['Minority %'] = data['Minority']/data['Population']
cols = ['raceeth_white_alone','raceeth_blackafricanamerican_alone','raceeth_americanindianalaskanative_alone','raceeth_asian_alone',
        'raceeth_nativehawaiianotherpacificislander_alone','raceeth_someotherrace_alone','raceeth_twoormoreraces_alone','raceeth_hispanicorlatino',
        'raceeth_whitealone_nothispanicorlatino']
data = data.drop(columns = cols)

### Age Brackets
+ M and F U5, 5 to 9, 10 to 14, 15 to 17, 18 to 24, 25 to 34, 35 to 44, 45 to 54, 55 to 64, 65 to 74, 75 to 84, 85+  
+ age brackets: under 18, 18 to 54, 55+  
+ 65+ 

In [140]:
#small groups m and f
data['Male Under 5'] = data['age_m_u5']
data['Female Under 5'] = data['age_f_u5']
data['Male 5 to 9'] = data['age_m_5to9']
data['Female 5 to 9'] = data['age_f_5to9']
data['Male 10 to 14'] = data['age_m_10to14']
data['Female 10 to 14'] = data['age_f_10to14']
data['Male 15 to 17'] = data['age_m_15to17']
data['Female 15 to 17'] = data['age_f_15to17']
data['Male 18 to 24'] = data['age_m_18to19']+data['age_m_20']+data['age_m_21']+data['age_m_22to24']
data['Female 18 to 24'] = data['age_f_18to19']+data['age_f_20']+data['age_f_21']+data['age_f_22to24']
data['Male 25 to 34'] = data['age_m_25to29']+data['age_m_30to34']
data['Female 25 to 34'] = data['age_f_25to29']+data['age_f_30to34']
data['Male 35 to 44'] = data['age_m_35to39']+data['age_m_40to44']
data['Female 35 to 44'] = data['age_f_35to39']+data['age_f_40to44']
data['Male 45 to 54'] = data['age_m_45to49']+data['age_m_50to54']
data['Female 45 to 54'] = data['age_f_45to49']+data['age_f_50to54']
data['Male 55 to 64'] = data['age_m_55to59']+data['age_m_60to61']+data['age_m_62to64']
data['Female 55 to 64'] = data['age_f_55to59']+data['age_f_60to61']+data['age_f_62to64']
data['Male 65 to 74'] = data['age_m_65to66']+data['age_m_67to69']+data['age_m_70to74']
data['Female 65 to 74'] = data['age_f_65to66']+data['age_f_67to69']+data['age_f_70to74']
data['Male 75 to 84'] = data['age_m_75to79']+data['age_m_80to84']
data['Female 75 to 84'] = data['age_f_75to79']+data['age_f_80to84']
data['Male 85 and Older'] = data['age_m_85+']
data['Female 85 and Older'] = data['age_f_85+']

In [141]:
#age brackets
u18list = [data['Male Under 5'],data['Female Under 5'],data['Male 5 to 9'],data['Female 5 to 9'],data['Male 10 to 14'],data['Female 10 to 14'],data['Male 15 to 17'],
           data['Female 15 to 17']]
data['Under 18'] = sum(u18list)
eighteento54list = [data['Male 18 to 24'],data['Female 18 to 24'],data['Male 25 to 34'],data['Female 25 to 34'],data['Male 35 to 44'],data['Female 35 to 44'],
              data['Male 45 to 54'],data['Female 45 to 54']]
data['18 to 24'] = sum(eighteento54list)
fifty5pluslist = [data['Male 55 to 64'],data['Female 55 to 64'],data['Male 65 to 74'],data['Female 65 to 74'],data['Male 75 to 84'],data['Female 75 to 84'],
                  data['Male 85 and Older'],data['Female 85 and Older']]
data['55 and Older'] = sum(fifty5pluslist)

In [142]:
#65+
sixty5pluslist = [data['Male 65 to 74'],data['Female 65 to 74'],data['Male 75 to 84'],data['Female 75 to 84'],data['Male 85 and Older'],data['Female 85 and Older']]
data['65 and Older'] = sum(sixty5pluslist)

In [143]:
cols = ['age_m_u5','age_m_5to9','age_m_10to14','age_m_15to17','age_m_18to19','age_m_20','age_m_21','age_m_22to24','age_m_25to29','age_m_30to34','age_m_35to39',
        'age_m_40to44','age_m_45to49','age_m_50to54','age_m_55to59','age_m_60to61','age_m_62to64','age_m_65to66','age_m_67to69','age_m_70to74','age_m_75to79',
        'age_m_80to84','age_m_85+','age_f_u5','age_f_5to9','age_f_10to14','age_f_15to17','age_f_18to19','age_f_20','age_f_21','age_f_22to24','age_f_25to29',
        'age_f_30to34','age_f_35to39','age_f_40to44','age_f_45to49','age_f_50to54','age_f_55to59','age_f_60to61','age_f_62to64','age_f_65to66','age_f_67to69',
        'age_f_70to74','age_f_75to79','age_f_80to84','age_f_85+']
data = data.drop(columns = cols)

In [144]:
data.head()

,NAME,age_total_male,age_total_female,hhsize_avg,healthcoverage_total_mallhealthcare,healthcoverage_mu6_all,healthcoverage_m6to18_all,healthcoverage_m19to25_all,healthcoverage_m26to34_all,healthcoverage_m35to44_all,healthcoverage_m45to54_all,healthcoverage_m55to64_all,healthcoverage_m65to74_all,healthcoverage_m75+_all,healthcoverage_total_fallhealthcare,healthcoverage_fu6_all,healthcoverage_f6to18_all,healthcoverage_f19to25_all,healthcoverage_f26to34_all,healthcoverage_f35to44_all,healthcoverage_f45to54_all,healthcoverage_f55to64_all,healthcoverage_f65to74_all,healthcoverage_f75+_all,healthcoverage_total_privatehealthcare_series,healthcoverage_total_mprivate,healthcoverage_mu6_privateall,healthcoverage_mu6_woprivate,healthcoverage_m6to18_privateall,healthcoverage_m6to18_woprivate,healthcoverage_m19to25_privateall,healthcoverage_m19to25_woprivate,healthcoverage_m26to34_privateall,healthcoverage_m26to34_woprivate,healthcoverage_m35to44_privateall,healthcoverage_m35to44_woprivate,healthcoverage_m45to54_privateall,healthcoverage_m45to54_woprivate,healthcoverage_m55to64_privateall,healthcoverage_m55to64_woprivate,healthcoverage_m65to74_privateall,healthcoverage_m65to74_woprivate,healthcoverage_m75+_privateall,healthcoverage_m75+_woprivate,healthcoverage_total_fprivate,healthcoverage_fu6_privateall,healthcoverage_fu6_woprivate,healthcoverage_f6to18_privateall,healthcoverage_f6to18_woprivate,healthcoverage_f19to25_privateall,healthcoverage_f19to25_woprivate,healthcoverage_f26to34_privateall,healthcoverage_f26to34_woprivate,healthcoverage_f35to44_privateall,healthcoverage_f35to44_woprivate,healthcoverage_f45to54_privateall,healthcoverage_f45to54_woprivate,healthcoverage_f55to64_privateall,healthcoverage_f55to64_woprivate,healthcoverage_f65to74_privateall,healthcoverage_f65to74_woprivate,healthcoverage_f75+_privateall,healthcoverage_f75+_woprivate,healthcoverage_total_publichealthcare_series,healthcoverage_total_mpublic,healthcoverage_mu6_publicall,healthcoverage_mu6_wopublic,healthcoverage_m6to18_publicall,healthcoverage_m6to18_wopublic,healthcoverage_m19to25_publicall,healthcoverage_m19to25_wopublic,healthcoverage_m26to34_publicall,healthcoverage_m26to34_wopublic,healthcoverage_m35to44_publicall,healthcoverage_m35to44_wopublic,healthcoverage_m45to54_publicall,healthcoverage_m45to54_wopublic,healthcoverage_m55to64_publicall,healthcoverage_m55to64_wopublic,healthcoverage_m65to74_publicall,healthcoverage_m65to74_wopublic,healthcoverage_m75+_publicall,healthcoverage_m75+_wopublic,healthcoverage_total_fpublic,healthcoverage_fu6_publicall,healthcoverage_fu6_wopublic,healthcoverage_f6to18_publicall,healthcoverage_f6to18_wopublic,healthcoverage_f19to25_publicall,healthcoverage_f19to25_wopublic,healthcoverage_f26to34_publicall,healthcoverage_f26to34_wopublic,healthcoverage_f35to44_publicall,healthcoverage_f35to44_wopublic,healthcoverage_f45to54_publicall,healthcoverage_f45to54_wopublic,healthcoverage_f55to64_publicall,healthcoverage_f55to64_wopublic,healthcoverage_f65to74_publicall,healthcoverage_f65to74_wopublic,healthcoverage_f75+_publicall,healthcoverage_f75+_wopublic,poverty_total_bysexbyage_series,poverty_belowlevel,units_allhousing,occupancy_total_series,occupancy_occupiedunits,occupancy_vacantunits,tenure_total_series,tenure_owneroccunits,tenure_renteroccunits,units_total_series,units_one_detached,units_one_attached,units_two,units_threetofour,units_fivetonine,units_tentonineteen,units_twentytofortynine,units_fiftyormore,units_mobilehome,units_boatrvvanetc,hhtype_total_series,hhtype_familyhh,hhtype_familyhh_marriedcouplefam,hhtype_familyhh_otherfam,hhtype_familyhh_malenospouse,hhtype_familyhh_femalenospouse,hhtype_nonfamhh,hhtype_nonfamhh_householderalone,hhtype_nonfamhh_householdernotalone,hhincome_median,housingcost_total_selectedownercosts%hhincome_series,housingcost_total%ownercostwmortgage_series,housingcost_%ownercost30to34.9_wmortgage,housingcost_%ownercost35to39.9_wmortgage,housingcost_%ownercost40to49.9_wmortgage,housingcost_%

In [145]:
data.tail()

,NAME,age_total_male,age_total_female,hhsize_avg,healthcoverage_total_mallhealthcare,healthcoverage_mu6_all,healthcoverage_m6to18_all,healthcoverage_m19to25_all,healthcoverage_m26to34_all,healthcoverage_m35to44_all,healthcoverage_m45to54_all,healthcoverage_m55to64_all,healthcoverage_m65to74_all,healthcoverage_m75+_all,healthcoverage_total_fallhealthcare,healthcoverage_fu6_all,healthcoverage_f6to18_all,healthcoverage_f19to25_all,healthcoverage_f26to34_all,healthcoverage_f35to44_all,healthcoverage_f45to54_all,healthcoverage_f55to64_all,healthcoverage_f65to74_all,healthcoverage_f75+_all,healthcoverage_total_privatehealthcare_series,healthcoverage_total_mprivate,healthcoverage_mu6_privateall,healthcoverage_mu6_woprivate,healthcoverage_m6to18_privateall,healthcoverage_m6to18_woprivate,healthcoverage_m19to25_privateall,healthcoverage_m19to25_woprivate,healthcoverage_m26to34_privateall,healthcoverage_m26to34_woprivate,healthcoverage_m35to44_privateall,healthcoverage_m35to44_woprivate,healthcoverage_m45to54_privateall,healthcoverage_m45to54_woprivate,healthcoverage_m55to64_privateall,healthcoverage_m55to64_woprivate,healthcoverage_m65to74_privateall,healthcoverage_m65to74_woprivate,healthcoverage_m75+_privateall,healthcoverage_m75+_woprivate,healthcoverage_total_fprivate,healthcoverage_fu6_privateall,healthcoverage_fu6_woprivate,healthcoverage_f6to18_privateall,healthcoverage_f6to18_woprivate,healthcoverage_f19to25_privateall,healthcoverage_f19to25_woprivate,healthcoverage_f26to34_privateall,healthcoverage_f26to34_woprivate,healthcoverage_f35to44_privateall,healthcoverage_f35to44_woprivate,healthcoverage_f45to54_privateall,healthcoverage_f45to54_woprivate,healthcoverage_f55to64_privateall,healthcoverage_f55to64_woprivate,healthcoverage_f65to74_privateall,healthcoverage_f65to74_woprivate,healthcoverage_f75+_privateall,healthcoverage_f75+_woprivate,healthcoverage_total_publichealthcare_series,healthcoverage_total_mpublic,healthcoverage_mu6_publicall,healthcoverage_mu6_wopublic,healthcoverage_m6to18_publicall,healthcoverage_m6to18_wopublic,healthcoverage_m19to25_publicall,healthcoverage_m19to25_wopublic,healthcoverage_m26to34_publicall,healthcoverage_m26to34_wopublic,healthcoverage_m35to44_publicall,healthcoverage_m35to44_wopublic,healthcoverage_m45to54_publicall,healthcoverage_m45to54_wopublic,healthcoverage_m55to64_publicall,healthcoverage_m55to64_wopublic,healthcoverage_m65to74_publicall,healthcoverage_m65to74_wopublic,healthcoverage_m75+_publicall,healthcoverage_m75+_wopublic,healthcoverage_total_fpublic,healthcoverage_fu6_publicall,healthcoverage_fu6_wopublic,healthcoverage_f6to18_publicall,healthcoverage_f6to18_wopublic,healthcoverage_f19to25_publicall,healthcoverage_f19to25_wopublic,healthcoverage_f26to34_publicall,healthcoverage_f26to34_wopublic,healthcoverage_f35to44_publicall,healthcoverage_f35to44_wopublic,healthcoverage_f45to54_publicall,healthcoverage_f45to54_wopublic,healthcoverage_f55to64_publicall,healthcoverage_f55to64_wopublic,healthcoverage_f65to74_publicall,healthcoverage_f65to74_wopublic,healthcoverage_f75+_publicall,healthcoverage_f75+_wopublic,poverty_total_bysexbyage_series,poverty_belowlevel,units_allhousing,occupancy_total_series,occupancy_occupiedunits,occupancy_vacantunits,tenure_total_series,tenure_owneroccunits,tenure_renteroccunits,units_total_series,units_one_detached,units_one_attached,units_two,units_threetofour,units_fivetonine,units_tentonineteen,units_twentytofortynine,units_fiftyormore,units_mobilehome,units_boatrvvanetc,hhtype_total_series,hhtype_familyhh,hhtype_familyhh_marriedcouplefam,hhtype_familyhh_otherfam,hhtype_familyhh_malenospouse,hhtype_familyhh_femalenospouse,hhtype_nonfamhh,hhtype_nonfamhh_householderalone,hhtype_nonfamhh_householdernotalone,hhincome_median,housingcost_total_selectedownercosts%hhincome_series,housingcost_total%ownercostwmortgage_series,housingcost_%ownercost30to34.9_wmortgage,housingcost_%ownercost35to39.9_wmortgage,housingcost_%ownercost40to49.9_wmortgage,housingcost_%

In [146]:
data.to_csv('../data/progress.csv')